<a href="https://colab.research.google.com/github/ximecaceres/materia-IA-/blob/main/ecommerce_cosmeticos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agente 1 para normalizar dataset


## SECCIÓN 1: CARGAR EL DATASET E IMPORTAR LIBRERÍAS

In [9]:
# Importamos la librería pandas, esencial para trabajar con DataFrames.
import pandas as pd

# Definimos la ruta del archivo CSV.
# Se corrige el nombre del archivo con un espacio extra, según lo que aparece en el sistema de archivos.
file_path = '/content/E-commerce  cosmetic dataset.csv'

# Intentamos cargar el archivo CSV utilizando la codificación UTF-8 como primera opción.
# Si hay un error de codificación, se puede intentar con 'latin1' o 'ISO-8859-1'.
try:
    df = pd.read_csv(file_path, encoding='utf-8')
    print("Dataset cargado exitosamente con codificación 'utf-8'.")
except UnicodeDecodeError:
    print("Error de codificación con 'utf-8'. Intentando con 'latin1'...")
    try:
        df = pd.read_csv(file_path, encoding='latin1')
        print("Dataset cargado exitosamente con codificación 'latin1'.")
    except Exception as e:
        print(f"No se pudo cargar el dataset con 'latin1' tampoco. Error: {e}")
        print("Por favor, verifique la codificación de su archivo CSV.")
        df = None # Aseguramos que df sea None si la carga falla

# Si el DataFrame se cargó correctamente, mostramos las primeras filas para una verificación inicial.
if df is not None:
    print("\nPrimeras 5 filas del dataset original:")
    display(df.head())

Error de codificación con 'utf-8'. Intentando con 'latin1'...
Dataset cargado exitosamente con codificación 'latin1'.

Primeras 5 filas del dataset original:


,product_name,website,country,category,subcategory,title-href,price,brand,ingredients,form,type,color,size,rating,noofratings
0,"Carlton London Incense Eau da parfum, Premium ...",Flipkart,India,body,perfume,https://www.amazon.in/Carlton-London-Limited-I...,599.0,Carlton London,NaN,aerosol,NaN,"Top Note: Orange Blossom, Blackberry | Heart N...",100,3.9,19
1,CHARLENE SPRAY MIST PERFUME 30 - INTIMATE (PAC...,Flipkart,India,body,perfume,https://www.amazon.in/CHARLENE-SPRAY-MIST-PERF...,149.0,Charlene,NaN,aerosol,NaN,Unit count type:,30,4.4,"4,031"
2,CHARLENE SPRAY MIST PERFUME 30 - INTIMATE (PAC...,Flipkart,India,body,perfume,https://www.amazon.in/CHARLENE-SPRAY-MIST-PERF...,298.0,Charlene,NaN,aerosol,NaN,Unit count type:,30,4.4,"4,072"
3,DENVER Black Code Perfume - 60 | Eau de Parfum...,Flipkart,India,body,perfume,https://www.amazon.in/DENVER-Black-Code-Perfum...,245.0,Denver,NaN,aerosol,NaN,Long-Lasting Scent,60,4.2,61
4,Denver Hamilton Perfume - 100 | Long Lasting P...,Flipkart,India,body,perfume,https://www.amazon.in/Denver-Perfume-Hamilton-...,422.0,Denver,NaN,aerosol,NaN,Long-Lasting Scent,100,4.3,342


In [10]:
print('Re-ejecutando la celda de carga de dataset con la ruta corregida.')

Re-ejecutando la celda de carga de dataset con la ruta corregida.


## SECCIÓN 2: NORMALIZACIÓN Y LIMPIEZA PASO A PASO

In [18]:
# 1. Normalizar nombres de columnas (snake_case)

# Función para convertir un string a snake_case
def to_snake_case(name):
    # Convierte a minúsculas
    name = name.lower()
    # Reemplaza espacios, guiones y otros caracteres no alfanuméricos con guiones bajos
    # Usa regex para ser más robusto
    name = pd.Series(name).str.replace(r'[^a-z0-9_]+', '_', regex=True).iloc[0]
    # Elimina guiones bajos al inicio o al final y reemplaza múltiples guiones bajos por uno solo
    name = pd.Series(name).str.strip('_').str.replace(r'__+', '_', regex=True).iloc[0]
    return name

# Aplicamos la función a todos los nombres de columnas del DataFrame
if df is not None:
    original_columns = df.columns.tolist()
    df.columns = [to_snake_case(col) for col in df.columns]
    print("\nNombres de columnas normalizados:")
    for original, new in zip(original_columns, df.columns):
        print(f"  '{original}' -> '{new}'")


Nombres de columnas normalizados:
  'product_name' -> 'product_name'
  'website' -> 'website'
  'country' -> 'country'
  'category' -> 'category'
  'subcategory' -> 'subcategory'
  'title-href' -> 'title_href'
  'price' -> 'price'
  'brand' -> 'brand'
  'ingredients' -> 'ingredients'
  'form' -> 'form'
  'type' -> 'type'
  'color' -> 'color'
  'size' -> 'size'
  'rating' -> 'rating'
  'noofratings' -> 'noofratings'


In [11]:
print('Re-ejecutando la celda de normalización de columnas.')

Re-ejecutando la celda de normalización de columnas.


In [19]:
# 2. Verificar valores nulos

# Calculamos el número de valores nulos por columna
if df is not None:
    null_counts = df.isnull().sum()
    print("\nValores nulos por columna (antes de la limpieza):")
    display(null_counts[null_counts > 0].to_frame(name='Nulos'))
    if null_counts.sum() == 0:
        print("No hay valores nulos en el dataset.")


Valores nulos por columna (antes de la limpieza):


,Nulos
price,317
ingredients,6015
type,2681
color,1989
size,3166
rating,2067
noofratings,459


In [12]:
print('Re-ejecutando la celda de verificación de nulos.')

Re-ejecutando la celda de verificación de nulos.


In [20]:
# 3. Reemplazar valores nulos

if df is not None:
    for col in df.columns:
        # Si la columna es numérica, rellenamos los nulos con 0
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].fillna(0)
            print(f"  Columna '{col}': valores nulos rellenados con 0.")
        # Si la columna es de tipo objeto (generalmente texto), rellenamos con 'no_definido'
        elif pd.api.types.is_object_dtype(df[col]):
            df[col] = df[col].fillna('no_definido')
            print(f"  Columna '{col}': valores nulos rellenados con 'no_definido'.")

    print("\nValores nulos después de rellenar:")
    display(df.isnull().sum().to_frame(name='Nulos'))

  Columna 'product_name': valores nulos rellenados con 'no_definido'.
  Columna 'website': valores nulos rellenados con 'no_definido'.
  Columna 'country': valores nulos rellenados con 'no_definido'.
  Columna 'category': valores nulos rellenados con 'no_definido'.
  Columna 'subcategory': valores nulos rellenados con 'no_definido'.
  Columna 'title_href': valores nulos rellenados con 'no_definido'.
  Columna 'price': valores nulos rellenados con 0.
  Columna 'brand': valores nulos rellenados con 'no_definido'.
  Columna 'ingredients': valores nulos rellenados con 'no_definido'.
  Columna 'form': valores nulos rellenados con 'no_definido'.
  Columna 'type': valores nulos rellenados con 'no_definido'.
  Columna 'color': valores nulos rellenados con 'no_definido'.
  Columna 'size': valores nulos rellenados con 'no_definido'.
  Columna 'rating': valores nulos rellenados con 'no_definido'.
  Columna 'noofratings': valores nulos rellenados con 'no_definido'.

Valores nulos después de rellen

,Nulos
product_name,0
website,0
country,0
category,0
subcategory,0
title_href,0
price,0
brand,0
ingredients,0
form,0


In [13]:
print('Re-ejecutando la celda de reemplazo de nulos.')

Re-ejecutando la celda de reemplazo de nulos.


In [21]:
# 4. Limpieza de texto (caracteres especiales rotos o extraños)

# Esta función limpia caracteres no ASCII y normaliza espacios en cadenas de texto.
def clean_text(text):
    if isinstance(text, str):
        # Elimina caracteres no ASCII (pueden ser 'rotos' o 'extraños')
        text = text.encode('ascii', 'ignore').decode('ascii')
        # Reemplaza múltiples espacios por uno solo
        text = ' '.join(text.split())
        # Elimina espacios al inicio y al final
        text = text.strip()
    return text

if df is not None:
    # Identificamos las columnas de tipo 'object' (que suelen contener texto)
    text_columns = df.select_dtypes(include='object').columns

    # Aplicamos la función de limpieza a cada columna de texto
    for col in text_columns:
        df[col] = df[col].apply(clean_text)
        print(f"  Columna '{col}': texto limpiado.")

    print("\nEjemplo de limpieza de texto (primeras filas con columnas de texto):")
    display(df[text_columns].head())

  Columna 'product_name': texto limpiado.
  Columna 'website': texto limpiado.
  Columna 'country': texto limpiado.
  Columna 'category': texto limpiado.
  Columna 'subcategory': texto limpiado.
  Columna 'title_href': texto limpiado.
  Columna 'brand': texto limpiado.
  Columna 'ingredients': texto limpiado.
  Columna 'form': texto limpiado.
  Columna 'type': texto limpiado.
  Columna 'color': texto limpiado.
  Columna 'size': texto limpiado.
  Columna 'rating': texto limpiado.
  Columna 'noofratings': texto limpiado.

Ejemplo de limpieza de texto (primeras filas con columnas de texto):


,product_name,website,country,category,subcategory,title_href,brand,ingredients,form,type,color,size,rating,noofratings
0,"Carlton London Incense Eau da parfum, Premium ...",Flipkart,India,body,perfume,https://www.amazon.in/Carlton-London-Limited-I...,Carlton London,no_definido,aerosol,no_definido,"Top Note: Orange Blossom, Blackberry | Heart N...",100,3.9,19
1,CHARLENE SPRAY MIST PERFUME 30 - INTIMATE (PAC...,Flipkart,India,body,perfume,https://www.amazon.in/CHARLENE-SPRAY-MIST-PERF...,Charlene,no_definido,aerosol,no_definido,Unit count type:,30,4.4,"4,031"
2,CHARLENE SPRAY MIST PERFUME 30 - INTIMATE (PAC...,Flipkart,India,body,perfume,https://www.amazon.in/CHARLENE-SPRAY-MIST-PERF...,Charlene,no_definido,aerosol,no_definido,Unit count type:,30,4.4,"4,072"
3,DENVER Black Code Perfume - 60 | Eau de Parfum...,Flipkart,India,body,perfume,https://www.amazon.in/DENVER-Black-Code-Perfum...,Denver,no_definido,aerosol,no_definido,Long-Lasting Scent,60,4.2,61
4,Denver Hamilton Perfume - 100 | Long Lasting P...,Flipkart,India,body,perfume,https://www.amazon.in/Denver-Perfume-Hamilton-...,Denver,no_definido,aerosol,no_definido,Long-Lasting Scent,100,4.3,342


In [14]:
print('Re-ejecutando la celda de limpieza de texto.')

Re-ejecutando la celda de limpieza de texto.


In [22]:
# 5. Mostrar estructura final

if df is not None:
    print(f"\nNúmero total de filas: {df.shape[0]}")
    print(f"Número total de columnas: {df.shape[1]}")
    print("\nVista previa (head) del DataFrame final después de la limpieza:")
    display(df.head())


Número total de filas: 12615
Número total de columnas: 15

Vista previa (head) del DataFrame final después de la limpieza:


,product_name,website,country,category,subcategory,title_href,price,brand,ingredients,form,type,color,size,rating,noofratings
0,"Carlton London Incense Eau da parfum, Premium ...",Flipkart,India,body,perfume,https://www.amazon.in/Carlton-London-Limited-I...,599.0,Carlton London,no_definido,aerosol,no_definido,"Top Note: Orange Blossom, Blackberry | Heart N...",100,3.9,19
1,CHARLENE SPRAY MIST PERFUME 30 - INTIMATE (PAC...,Flipkart,India,body,perfume,https://www.amazon.in/CHARLENE-SPRAY-MIST-PERF...,149.0,Charlene,no_definido,aerosol,no_definido,Unit count type:,30,4.4,"4,031"
2,CHARLENE SPRAY MIST PERFUME 30 - INTIMATE (PAC...,Flipkart,India,body,perfume,https://www.amazon.in/CHARLENE-SPRAY-MIST-PERF...,298.0,Charlene,no_definido,aerosol,no_definido,Unit count type:,30,4.4,"4,072"
3,DENVER Black Code Perfume - 60 | Eau de Parfum...,Flipkart,India,body,perfume,https://www.amazon.in/DENVER-Black-Code-Perfum...,245.0,Denver,no_definido,aerosol,no_definido,Long-Lasting Scent,60,4.2,61
4,Denver Hamilton Perfume - 100 | Long Lasting P...,Flipkart,India,body,perfume,https://www.amazon.in/Denver-Perfume-Hamilton-...,422.0,Denver,no_definido,aerosol,no_definido,Long-Lasting Scent,100,4.3,342


In [15]:
print('Re-ejecutando la celda de mostrar estructura final.')

Re-ejecutando la celda de mostrar estructura final.


## SECCIÓN 3: GUARDAR EL RESULTADO

In [23]:
# Definimos el nombre del nuevo archivo CSV limpio
output_file_path = 'cosmetics_limpio.csv'

if df is not None:
    # Exportamos el DataFrame limpio a un nuevo archivo CSV
    # index=False evita que pandas escriba el índice del DataFrame como una columna en el CSV
    df.to_csv(output_file_path, index=False, encoding='utf-8')
    print(f"\nDataFrame limpio guardado exitosamente en '{output_file_path}'.")


DataFrame limpio guardado exitosamente en 'cosmetics_limpio.csv'.


In [16]:
print('Re-ejecutando la celda de guardar resultado.')

Re-ejecutando la celda de guardar resultado.


## VERIFICACIÓN DEL ARCHIVO GENERADO

In [24]:
# Cargamos el archivo recién guardado para verificar su integridad
try:
    df_cleaned_check = pd.read_csv(output_file_path, encoding='utf-8')
    print(f"\n'{output_file_path}' cargado para verificación.")

    # Verificamos que la suma total de valores nulos sea 0
    total_nulls_check = df_cleaned_check.isnull().sum().sum()

    print(f"Número total de valores nulos en '{output_file_path}': {total_nulls_check}")

    if total_nulls_check == 0:
        print("¡Confirmación exitosa! La suma de nulos en el archivo limpio es estrictamente 0.")
        print("Primeras 5 filas del archivo limpio verificado:")
        display(df_cleaned_check.head())
    else:
        print("Advertencia: Aún existen valores nulos en el archivo limpio. Por favor, revise el proceso de limpieza.")
        print("Valores nulos restantes por columna:")
        display(df_cleaned_check.isnull().sum()[df_cleaned_check.isnull().sum() > 0].to_frame(name='Nulos'))

except Exception as e:
    print(f"Error al cargar o verificar el archivo '{output_file_path}'. Error: {e}")


'cosmetics_limpio.csv' cargado para verificación.
Número total de valores nulos en 'cosmetics_limpio.csv': 0
¡Confirmación exitosa! La suma de nulos en el archivo limpio es estrictamente 0.
Primeras 5 filas del archivo limpio verificado:


,product_name,website,country,category,subcategory,title_href,price,brand,ingredients,form,type,color,size,rating,noofratings
0,"Carlton London Incense Eau da parfum, Premium ...",Flipkart,India,body,perfume,https://www.amazon.in/Carlton-London-Limited-I...,599.0,Carlton London,no_definido,aerosol,no_definido,"Top Note: Orange Blossom, Blackberry | Heart N...",100,3.9,19
1,CHARLENE SPRAY MIST PERFUME 30 - INTIMATE (PAC...,Flipkart,India,body,perfume,https://www.amazon.in/CHARLENE-SPRAY-MIST-PERF...,149.0,Charlene,no_definido,aerosol,no_definido,Unit count type:,30,4.4,"4,031"
2,CHARLENE SPRAY MIST PERFUME 30 - INTIMATE (PAC...,Flipkart,India,body,perfume,https://www.amazon.in/CHARLENE-SPRAY-MIST-PERF...,298.0,Charlene,no_definido,aerosol,no_definido,Unit count type:,30,4.4,"4,072"
3,DENVER Black Code Perfume - 60 | Eau de Parfum...,Flipkart,India,body,perfume,https://www.amazon.in/DENVER-Black-Code-Perfum...,245.0,Denver,no_definido,aerosol,no_definido,Long-Lasting Scent,60,4.2,61
4,Denver Hamilton Perfume - 100 | Long Lasting P...,Flipkart,India,body,perfume,https://www.amazon.in/Denver-Perfume-Hamilton-...,422.0,Denver,no_definido,aerosol,no_definido,Long-Lasting Scent,100,4.3,342


In [17]:
print('Re-ejecutando la celda de verificación final.')

Re-ejecutando la celda de verificación final.
